In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [22, 25, np.nan, 30, 35, 100, np.nan, 28],
    "Monthly_Income": [20000, 25000, 30000, np.nan,
                       35000, 500000, 28000, np.nan],
    "City": ["Delhi", "Mumbai", "Delhi", np.nan,
             "Pune", "Delhi", np.nan, "Mumbai"],
    "Purchase_Category": ["Electronics", "Clothing", np.nan,
                          "Electronics", "Clothing", "Electronics",
                          "Clothing", np.nan]
})

print(df)

     Age  Monthly_Income    City Purchase_Category
0   22.0         20000.0   Delhi       Electronics
1   25.0         25000.0  Mumbai          Clothing
2    NaN         30000.0   Delhi               NaN
3   30.0             NaN     NaN       Electronics
4   35.0         35000.0    Pune          Clothing
5  100.0        500000.0   Delhi       Electronics
6    NaN         28000.0     NaN          Clothing
7   28.0             NaN  Mumbai               NaN


Q1(A) Identify missing values and apply mean, median, and mode imputation

In [2]:
# Check missing values in each column
print("Missing Values:")
print(df.isnull().sum())

# Create separate datasets
mean_df = df.copy()
median_df = df.copy()

# Mean imputation for numerical columns
mean_df["Age"] = mean_df["Age"].fillna(mean_df["Age"].mean())
mean_df["Monthly_Income"] = mean_df["Monthly_Income"].fillna(
    mean_df["Monthly_Income"].mean()
)

# Median imputation for numerical columns
median_df["Age"] = median_df["Age"].fillna(median_df["Age"].median())
median_df["Monthly_Income"] = median_df["Monthly_Income"].fillna(
    median_df["Monthly_Income"].median()
)

# Mode imputation for categorical columns
for col in ["City", "Purchase_Category"]:
    mode_value = df[col].mode()[0]
    mean_df[col] = mean_df[col].fillna(mode_value)
    median_df[col] = median_df[col].fillna(mode_value)

print("\nDataset after Mean Imputation:")
print(mean_df)

print("\nDataset after Median Imputation:")
print(median_df)

Missing Values:
Age                  2
Monthly_Income       2
City                 2
Purchase_Category    2
dtype: int64

Dataset after Mean Imputation:
     Age  Monthly_Income    City Purchase_Category
0   22.0    20000.000000   Delhi       Electronics
1   25.0    25000.000000  Mumbai          Clothing
2   40.0    30000.000000   Delhi          Clothing
3   30.0   106333.333333   Delhi       Electronics
4   35.0    35000.000000    Pune          Clothing
5  100.0   500000.000000   Delhi       Electronics
6   40.0    28000.000000   Delhi          Clothing
7   28.0   106333.333333  Mumbai          Clothing

Dataset after Median Imputation:
     Age  Monthly_Income    City Purchase_Category
0   22.0         20000.0   Delhi       Electronics
1   25.0         25000.0  Mumbai          Clothing
2   29.0         30000.0   Delhi          Clothing
3   30.0         29000.0   Delhi       Electronics
4   35.0         35000.0    Pune          Clothing
5  100.0        500000.0   Delhi       Electroni

Q1(B)

In [3]:
# Numerical columns
numeric_cols = ["Age", "Monthly_Income"]

# Function to calculate statistics
def calculate_statistics(data, name):
    print("\n", name)
    print(data[numeric_cols].agg(["mean", "median", "std"]))

# Original dataset
calculate_statistics(df, "Before Imputation")

# After mean imputation
calculate_statistics(mean_df, "After Mean Imputation")

# After median imputation
calculate_statistics(median_df, "After Median Imputation")


 Before Imputation
             Age  Monthly_Income
mean    40.00000   106333.333333
median  29.00000    29000.000000
std     29.72541   192921.400230

 After Mean Imputation
              Age  Monthly_Income
mean    40.000000   106333.333333
median  32.500000    32500.000000
std     25.122557   163048.342232

 After Median Imputation
              Age  Monthly_Income
mean    37.250000    87000.000000
median  29.000000    29000.000000
std     25.633405   166931.979304


Question 2: Implementing SimpleImputer with Different Strategies

In [4]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Patient_Age": [25, 32, np.nan, 45, 52, np.nan, 38, 60],
    "Blood_Sugar": [90, 110, 105, np.nan, 250, 130, np.nan, 115],
    "Blood_Group": ["A", "B", "O", np.nan, "A", "B", np.nan, "A"],
    "Treatment": ["Basic", np.nan, "Advanced", "Basic",
                  "Advanced", "Basic", np.nan, "Basic"]
})

print(df)

   Patient_Age  Blood_Sugar Blood_Group Treatment
0         25.0         90.0           A     Basic
1         32.0        110.0           B       NaN
2          NaN        105.0           O  Advanced
3         45.0          NaN         NaN     Basic
4         52.0        250.0           A  Advanced
5          NaN        130.0           B     Basic
6         38.0          NaN         NaN       NaN
7         60.0        115.0           A     Basic


Q2(A)

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Separate numerical and categorical columns
numeric_cols = ["Patient_Age", "Blood_Sugar"]
categorical_cols = ["Blood_Group", "Treatment"]

# Create preprocessing workflow
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), ["Patient_Age"]),
        ("sugar", SimpleImputer(strategy="median"), ["Blood_Sugar"]),
        ("cat", SimpleImputer(strategy="most_frequent"), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Apply imputation
result = preprocessor.fit_transform(df)

# Convert into DataFrame with original column names
completed_df = pd.DataFrame(
    result,
    columns=numeric_cols + categorical_cols,
    index=df.index
)

# Convert numerical columns back to numeric datatype
completed_df[numeric_cols] = completed_df[numeric_cols].astype(float)

print("Completed Dataset:")
print(completed_df)

# Verify missing values
print("\nMissing Values After Imputation:")
print(completed_df.isnull().sum())

Completed Dataset:
   Patient_Age  Blood_Sugar Blood_Group Treatment
0         25.0         90.0           A     Basic
1         32.0        110.0           B     Basic
2         42.0        105.0           O  Advanced
3         45.0        112.5           A     Basic
4         52.0        250.0           A  Advanced
5         42.0        130.0           B     Basic
6         38.0        112.5           A     Basic
7         60.0        115.0           A     Basic

Missing Values After Imputation:
Patient_Age    0
Blood_Sugar    0
Blood_Group    0
Treatment      0
dtype: int64


Q2(B)

In [6]:
# Create a copy of original dataset
constant_df = df.copy()

# Separate numerical and categorical columns
numeric_cols = ["Patient_Age", "Blood_Sugar"]
categorical_cols = ["Blood_Group", "Treatment"]

# Create constant imputers
numeric_imputer = SimpleImputer(strategy="constant", fill_value=0)
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

# Apply numerical imputation
constant_df[numeric_cols] = numeric_imputer.fit_transform(
    constant_df[numeric_cols]
)

# Apply categorical imputation
constant_df[categorical_cols] = categorical_imputer.fit_transform(
    constant_df[categorical_cols]
)

print("Dataset after Constant Imputation:")
print(constant_df)

print("\nMissing Values:")
print(constant_df.isnull().sum())

Dataset after Constant Imputation:
   Patient_Age  Blood_Sugar Blood_Group Treatment
0         25.0         90.0           A     Basic
1         32.0        110.0           B   Unknown
2          0.0        105.0           O  Advanced
3         45.0          0.0     Unknown     Basic
4         52.0        250.0           A  Advanced
5          0.0        130.0           B     Basic
6         38.0          0.0     Unknown   Unknown
7         60.0        115.0           A     Basic

Missing Values:
Patient_Age    0
Blood_Sugar    0
Blood_Group    0
Treatment      0
dtype: int64


Question 3: Investigating the Impact of Outliers on Imputation

In [7]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer_ID": range(1, 11),
    "Annual_Spending": [12000, 15000, np.nan, 18000, 22000,
                        25000, 30000, np.nan, 900000, 20000],
    "Order_Count": [4, 5, 6, np.nan, 8, 7, 9, 10, 50, np.nan],
    "Membership": ["Silver", "Gold", "Silver", "Gold", np.nan,
                   "Silver", "Gold", np.nan, "Platinum", "Silver"]
})

print(df)

   Customer_ID  Annual_Spending  Order_Count Membership
0            1          12000.0          4.0     Silver
1            2          15000.0          5.0       Gold
2            3              NaN          6.0     Silver
3            4          18000.0          NaN       Gold
4            5          22000.0          8.0        NaN
5            6          25000.0          7.0     Silver
6            7          30000.0          9.0       Gold
7            8              NaN         10.0        NaN
8            9         900000.0         50.0   Platinum
9           10          20000.0          NaN     Silver


Q3(A)

In [8]:
from sklearn.impute import SimpleImputer

# Select numerical columns
numeric_cols = ["Annual_Spending", "Order_Count"]

# Mean imputation
mean_imputer = SimpleImputer(strategy="mean")
mean_data = mean_imputer.fit_transform(df[numeric_cols])

mean_df = pd.DataFrame(
    mean_data,
    columns=numeric_cols,
    index=df.index
)

# Median imputation
median_imputer = SimpleImputer(strategy="median")
median_data = median_imputer.fit_transform(df[numeric_cols])

median_df = pd.DataFrame(
    median_data,
    columns=numeric_cols,
    index=df.index
)

print("After Mean Imputation:")
print(mean_df)

print("\nAfter Median Imputation:")
print(median_df)

# Compare Annual Spending statistics
print("\nMean Imputation Statistics:")
print(mean_df["Annual_Spending"].agg(["mean", "std"]))

print("\nMedian Imputation Statistics:")
print(median_df["Annual_Spending"].agg(["mean", "std"]))

After Mean Imputation:
   Annual_Spending  Order_Count
0          12000.0        4.000
1          15000.0        5.000
2         130250.0        6.000
3          18000.0       12.375
4          22000.0        8.000
5          25000.0        7.000
6          30000.0        9.000
7         130250.0       10.000
8         900000.0       50.000
9          20000.0       12.375

After Median Imputation:
   Annual_Spending  Order_Count
0          12000.0          4.0
1          15000.0          5.0
2          21000.0          6.0
3          18000.0          7.5
4          22000.0          8.0
5          25000.0          7.0
6          30000.0          9.0
7          21000.0         10.0
8         900000.0         50.0
9          20000.0          7.5

Mean Imputation Statistics:
mean    130250.000000
std     274343.964962
Name: Annual_Spending, dtype: float64

Median Imputation Statistics:
mean    108400.000000
std     278184.271462
Name: Annual_Spending, dtype: float64


Q3(B)

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Exclude Customer_ID
X = df.drop(columns=["Customer_ID"])

# Define numerical and categorical columns
numeric_cols = ["Annual_Spending", "Order_Count"]
categorical_cols = ["Membership"]

# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), categorical_cols)
    ],
    verbose_feature_names_out=False
)

# Fit and transform the dataset
transformed_data = preprocessor.fit_transform(X)

# Get feature names
feature_names = preprocessor.get_feature_names_out()

# Convert into DataFrame
final_df = pd.DataFrame(
    transformed_data,
    columns=feature_names,
    index=df.index
)

# Convert numerical columns into numeric datatype
final_df[numeric_cols] = final_df[numeric_cols].astype(float)

print("Final Preprocessed Dataset:")
print(final_df)

# Verify missing values
print("\nMissing Values:")
print(final_df.isnull().sum())

# Verify feature columns
print("\nFeature Columns:")
print(final_df.columns.tolist())

# Verify dataset dimensions
print("\nDataset Shape:")
print(final_df.shape)

Final Preprocessed Dataset:
   Annual_Spending  Order_Count Membership
0          12000.0          4.0     Silver
1          15000.0          5.0       Gold
2          21000.0          6.0     Silver
3          18000.0          7.5       Gold
4          22000.0          8.0     Silver
5          25000.0          7.0     Silver
6          30000.0          9.0       Gold
7          21000.0         10.0     Silver
8         900000.0         50.0   Platinum
9          20000.0          7.5     Silver

Missing Values:
Annual_Spending    0
Order_Count        0
Membership         0
dtype: int64

Feature Columns:
['Annual_Spending', 'Order_Count', 'Membership']

Dataset Shape:
(10, 3)
